# TOKATÇI B2C — MAIN-ağırlıklı Kaggle eğitimi

Bu notebook CPU'da eğitim yapmaz. **GPU** ister.

- Girdi: Kaggle’daki **tokatci_data** (`dataset_tokatci_b2c.npz`, `deck_top2.csv`) ve **tokatci-code** (`src/mazlum/train_bc.py`) datasetleri.
- Yaptığı iş: B2C verisinde `context == 0` (MAIN) satırlarını **2.5x** ağırlıklandırır.
- Model: mevcut `MazlumNet` (B2C ile aynı mimari).
- Her epoch checkpoint kaydedilir; en iyi model **val MAIN doğruluğuna** göre seçilir.
- Çıktı: `/kaggle/working/model_tokatci_b2c_main25_best.npz`.

> Not: Bu model tek başına “lethal override”ı açmaz. Deploy ederken `bots/tokatci_b2c_v2/agent.py` içindeki `self.use_lethal = False` ayarını koru. 600 BO3 doğrulaması yapılmadan submit etme.


In [ ]:
from pathlib import Path
import json, shutil, sys, zipfile
import numpy as np

INPUT = Path('/kaggle/input')
WORK = Path('/kaggle/working')
WORK.mkdir(exist_ok=True)

# Kaggle ekran görüntüsündeki iki dataset:
#   /kaggle/input/tokatci_data/
#   /kaggle/input/tokatci-code/
# Alt çizgi/tire farkı olursa diye adayları tek tek deniyoruz.
def first_with(rel, candidates):
    for base in candidates:
        if (base / rel).is_file():
            return base
    return None

DATA_DIR = first_with('dataset_tokatci_b2c.npz', [
    Path('/kaggle/input/tokatci_data'),
    Path('/kaggle/input/tokatci-data'),
])
CODE = first_with('src/mazlum/train_bc.py', [
    Path('/kaggle/input/tokatci-code'),
    Path('/kaggle/input/tokatci_code'),
])

# Beklenen yollar yoksa zip'i aç (eski upload biçimi için fallback).
if DATA_DIR is None:
    hits = list(INPUT.rglob('dataset_tokatci_b2c.npz'))
    if hits:
        DATA_DIR = hits[0].parent
    else:
        z = next(iter(INPUT.rglob('tokatci_datasets.zip')), None)
        if z is None:
            raise FileNotFoundError('tokatci_data/dataset_tokatci_b2c.npz bulunamadı')
        dest = WORK / 'tokatci_datasets'
        if dest.exists(): shutil.rmtree(dest)
        dest.mkdir()
        with zipfile.ZipFile(z) as f: f.extractall(dest)
        DATA_DIR = next(dest.rglob('dataset_tokatci_b2c.npz')).parent

if CODE is None:
    hits = [x for x in INPUT.rglob('train_bc.py') if x.as_posix().endswith('src/mazlum/train_bc.py')]
    if hits:
        # .../src/mazlum/train_bc.py -> dataset kökü
        CODE = hits[0].parents[2]
    else:
        z = next(iter(INPUT.rglob('tokatci_code.zip')), None)
        if z is None:
            raise FileNotFoundError('tokatci-code/src/mazlum/train_bc.py bulunamadı')
        dest = WORK / 'tokatci_code'
        if dest.exists(): shutil.rmtree(dest)
        dest.mkdir()
        with zipfile.ZipFile(z) as f: f.extractall(dest)
        CODE = dest

DATA = DATA_DIR / 'dataset_tokatci_b2c.npz'
DECK = DATA_DIR / 'deck_top2.csv'
print('DATA_DIR =', DATA_DIR)
print('DATA     =', DATA, DATA.is_file())
print('DECK     =', DECK, DECK.is_file())
print('CODE     =', CODE)
print('train_bc =', (CODE / 'src/mazlum/train_bc.py').is_file())

assert DATA.is_file(), DATA
assert (CODE / 'src/mazlum/train_bc.py').is_file(), CODE


In [ ]:
# MAIN-ağırlıklı veri seti. İstersen MAIN_WEIGHT değerini değiştir.
MAIN_WEIGHT = 2.5
weighted = WORK/'dataset_tokatci_b2c_main25.npz'
WOUT = WORK/'checkpoints_b2c_main25'
if WOUT.exists(): shutil.rmtree(WOUT)

d = np.load(DATA)
w = d['sample_weight'].astype(np.float32).copy()
ctx = d['context'].astype(np.int16)
w = np.where(ctx == 0, w * MAIN_WEIGHT, w)
arrays = {k: d[k] for k in d.files if k != 'sample_weight'}
np.savez_compressed(weighted, **arrays, sample_weight=w)
print('yazdı:', weighted, f'{weighted.stat().st_size/1e6:.1f} MB')
print('MAIN satır:', int((ctx==0).sum()), '/', len(ctx), 'ağırlık ort:', float(w[ctx==0].mean()))


In [ ]:
import subprocess
OUT = WORK/'model_tokatci_b2c_main25.npz'
cmd = [
    sys.executable, str(CODE/'src/mazlum/train_bc.py'),
    '--dataset', str(weighted), '--out', str(OUT),
    '--device', 'cuda', '--batch-size', '256',
    '--epochs', '35', '--patience', '8',
    '--dropout', '0.10', '--lr', '1e-3',
    '--arena-every', '0', '--select-metric', 'val',
    '--save-each', str(WOUT),
    '--print-every', '1', '--seed', '45',
]
print(' '.join(cmd))
subprocess.run(cmd, check=True)
print('final checkpoint:', OUT)


In [ ]:
# Checkpointleri val MAIN doğruluğuna göre seç.
import torch
sys.path.insert(0, str(CODE/'src/mazlum'))
import model as M

dd = np.load(DATA)
val = np.nonzero(dd['split'] == 1)[0]
y = dd['label'][val]
ctx = dd['context'][val]
typ = dd['selected_type'][val]
mc = dd['max_count'][val]
main = ctx == 0

def load_ckpt(path):
    m = np.load(path)
    net = M.MazlumNet()
    sd = {k: torch.from_numpy(m[k].astype(np.float32)) for k in m.files
          if k not in ('state_mean','state_std','option_mean','option_std','temperature')}
    net.load_state_dict(sd)
    net.eval().cuda()
    norm = {k: torch.from_numpy(m[k].astype(np.float32)).cuda()
            for k in ('state_mean','state_std','option_mean','option_std')}
    return net, norm

@torch.no_grad()
def predict(net, norm, idx, bs=4096):
    out = []
    for start in range(0, len(idx), bs):
        ii = idx[start:start+bs]
        s = torch.from_numpy(dd['state'][ii].astype(np.float32)).cuda()
        o = torch.from_numpy(dd['options'][ii].astype(np.float32)).cuda()
        c = torch.from_numpy(dd['card_ids'][ii].astype(np.int64)).cuda()
        m = torch.from_numpy(dd['mask'][ii].astype(np.float32)).cuda()
        s = (s - norm['state_mean']) / norm['state_std']
        o = (o - norm['option_mean']) / norm['option_std']
        out.append(net(s, o, c, m).argmax(-1).cpu().numpy())
    return np.concatenate(out)

rows = []
for ck in sorted(WOUT.glob('epoch_*.npz')):
    net, norm = load_ckpt(ck)
    p = predict(net, norm, val)
    rows.append({
        'checkpoint': ck.name,
        'overall': float((p == y).mean()),
        'main': float((p[main] == y[main]).mean()),
        'attach': float((p[typ==8] == y[typ==8]).mean()) if (typ==8).any() else 0.0,
        'retreat': float((p[typ==12] == y[typ==12]).mean()) if (typ==12).any() else 0.0,
        'single_main': float((p[main & (mc==1)] == y[main & (mc==1)]).mean()),
    })
    print(rows[-1])

if not rows:
    raise RuntimeError('checkpoint yok; eğitim tamamlanmadı?')

# Seçim: önce main, sonra overall.
best = max(rows, key=lambda r: (r['main'], r['overall']))
print('BEST', best)
shutil.copy(WOUT/best['checkpoint'], WORK/'model_tokatci_b2c_main25_best.npz')
(WORK/'b2c_main25_metrics.json').write_text(json.dumps(rows, indent=2))
print('seçilen model:', WORK/'model_tokatci_b2c_main25_best.npz')


In [ ]:
# İndirme/taşıma özeti.
for name in ['model_tokatci_b2c_main25_best.npz', 'b2c_main25_metrics.json']:
    p = Path('/kaggle/working')/name
    print(name, p.exists(), p.stat().st_size/1024 if p.exists() else 0, 'KB')

print("Yerelde:")
print("  cp /kaggle/working/model_tokatci_b2c_main25_best.npz \\")
print("     bots/tokatci_b2c_v2/model.npz")
print("  # bots/tokatci_b2c_v2/agent.py içinde self.use_lethal = False kalsın")
print("  # 100 BO3 smoke -> 600 BO3 doğrulama; geçmeden submit yok.")
